# climagrid: deep learning vs gradient boosting (Kaggle)

**Question this notebook answers:** does a deep temporal model (an LSTM) forecast asset stress more accurately than the shipped LightGBM model, judged in the *same* honest backtest harness?

**Honest premise.** This dataset is small (a few dozen assets, daily resolution). On small tabular data, gradient-boosted trees usually match or beat neural nets, so the likely outcome is the LSTM ties or slightly loses at short horizons and *maybe* wins at longer ones or on the active (event) periods. The deliverable is the rigorous comparison, not a foregone 'deep learning wins'. We report both models through identical splits, the same quantiles, the same conformal calibration, multi-seed spread for the LSTM (its run-to-run noise can exceed the gap), and the compute cost of each. Whichever wins is the one worth shipping.

## Setup

In [ ]:
%pip install -q "climagrid[dl] @ git+https://github.com/TemidireAdesiji/climagrid@feat/forecasting-dl"

In [ ]:
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import climagrid
from climagrid.forecasting import ForecastConfig, evaluate
from climagrid.forecasting.config import LSTMParams
from climagrid.forecasting.dataset import build_supervised_frame, build_training_panel
from climagrid.forecasting.models import LightGBMForecaster
from climagrid.forecasting.torch_models import LSTMForecaster

warnings.filterwarnings("ignore")
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("climagrid_out")
CACHE = OUT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("climagrid", climagrid.__version__, "| torch", torch.__version__, "| device:", DEVICE)
print("output dir:", OUT)

## 1. Assets and configuration

Both models read the *same* supervised frame, so they see identical information and the comparison is fair. The only requirement for the LSTM is **contiguous lags** (`lags = 1..L`) so its encoder sequence is a true daily window with no gaps; LightGBM is indifferent to this, so it is a fair shared choice.

In [ ]:
import os

ASSETS_CSV = "your_assets.csv"  # <-- your CSV (asset_id, lat, lon)
if not os.path.exists(ASSETS_CSV):
    local = Path(climagrid.__file__).resolve().parents[2] / "examples" / "data" / "sample_assets.csv"
    if local.exists():
        ASSETS_CSV = str(local)
    else:
        url = "https://raw.githubusercontent.com/TemidireAdesiji/climagrid/main/examples/data/sample_assets.csv"
        ASSETS_CSV = str(OUT / "sample_assets.csv")
        pd.read_csv(url).to_csv(ASSETS_CSV, index=False)
print(len(pd.read_csv(ASSETS_CSV)), "assets from", ASSETS_CSV)

In [ ]:
# Keep the benchmark focused: one strong instantaneous target, one strong, one
# seasonal-rolling target. Add more at the cost of runtime.
TARGETS = [
    "feat_thermal_aging_factor",   # IEEE C57.91, instantaneous - the headline target
    "feat_conductor_sag_index",    # IEEE 738, instantaneous - strong
    "feat_freeze_thaw_cycles",     # 720h rolling - seasonal, persistence is a hard baseline
]
HORIZON = 7
N_SPLITS, TEST_SIZE = 3, 90
N_SEEDS = 5          # LSTM seeds (report mean +/- std); set 1 for a quick pass
HISTORY_YEARS = 15   # both models use the same history window

# Contiguous 28-day lag window -> a clean LSTM encoder sequence; same frame for LightGBM.
LSTM = LSTMParams(
    hidden_size=64, num_layers=1, dropout=0.1,
    epochs=60, batch_size=256, learning_rate=1e-3, patience=8, device="auto",
)
config = ForecastConfig(
    targets=TARGETS, horizon_days=HORIZON,
    lags=list(range(1, 29)), rolling_windows=[7, 30], quantiles=[0.1, 0.5, 0.9],
    history_years=HISTORY_YEARS, calibrate_intervals=True,  # mondrian conformal
    cache_dir=CACHE, lstm=LSTM,
)
END = datetime(2025, 12, 31, tzinfo=timezone.utc)
START = datetime(END.year - HISTORY_YEARS, 1, 1, tzinfo=timezone.utc)
print("history:", START.date(), "->", END.date(), "| seeds:", N_SEEDS)

## 2. Build the daily panel once (cached)

The fetch is the slow step. It is cached to parquet (and saved for download) so re-runs skip the network.

In [ ]:
panel = build_training_panel(ASSETS_CSV, START, END, config)
if panel.empty:
    raise RuntimeError("Empty panel: on Kaggle enable Internet (Settings -> Internet).")
panel.to_parquet(OUT / "daily_panel_dl.parquet", index=False)
present = [t for t in TARGETS if t in panel.columns]
print("panel:", panel.shape, "| factors present:", len(present), present)
panel.head()

## 3. Backtest both models in the identical harness

`evaluate(...)` runs the rolling-origin backtest with the embargo gap and scores against persistence and climatology. The only thing that changes between the two runs is `model_factory`. LightGBM is deterministic (fixed `random_state`); the LSTM is run across `N_SEEDS` because its run-to-run variance is real and must be reported, not hidden.

In [ ]:
t0 = time.time()
lgbm = evaluate(panel, config, n_splits=N_SPLITS, test_size_days=TEST_SIZE)
lgbm_secs = time.time() - t0
lgbm["model"] = "lightgbm"; lgbm["seed"] = config.random_state
print(f"LightGBM backtest: {lgbm_secs:.1f}s")

lstm_runs = []
lstm_secs = 0.0
for s in range(N_SEEDS):
    cfg_s = config.model_copy(update={"lstm": config.lstm.model_copy(update={"seed": s})})
    t0 = time.time()
    sc = evaluate(panel, cfg_s, n_splits=N_SPLITS, test_size_days=TEST_SIZE,
                  model_factory=LSTMForecaster)
    lstm_secs += time.time() - t0
    sc["model"] = "lstm"; sc["seed"] = s
    lstm_runs.append(sc)
    print(f"  LSTM seed {s}: cumulative {lstm_secs:.1f}s")
lstm = pd.concat(lstm_runs, ignore_index=True)
print(f"LSTM backtest ({N_SEEDS} seeds): {lstm_secs:.1f}s total, {lstm_secs/N_SEEDS:.1f}s/seed")

## 4. Head-to-head comparison table

Per target and horizon: mean skill vs persistence (higher is better), the LSTM's seed spread, interval coverage (target 0.80), and pinball loss. A win only counts if it clears the LSTM's own seed noise band.

In [ ]:
METRICS = ["skill_vs_persistence", "skill_vs_persistence_events",
           "interval_coverage", "pinball"]

# Average folds first; for the LSTM also average within a seed, then take the
# mean and std ACROSS seeds so the spread reflects training randomness.
lgbm_ph = lgbm.groupby(["target", "horizon_day"])[METRICS].mean()
lstm_seed = lstm.groupby(["seed", "target", "horizon_day"])[METRICS].mean().reset_index()
lstm_mean = lstm_seed.groupby(["target", "horizon_day"])[METRICS].mean()
lstm_std = lstm_seed.groupby(["target", "horizon_day"])["skill_vs_persistence"].std()

table = pd.DataFrame({
    "skill_lgbm": lgbm_ph["skill_vs_persistence"],
    "skill_lstm": lstm_mean["skill_vs_persistence"],
    "skill_lstm_std": lstm_std,
    "cover_lgbm": lgbm_ph["interval_coverage"],
    "cover_lstm": lstm_mean["interval_coverage"],
    "pinball_lgbm": lgbm_ph["pinball"],
    "pinball_lstm": lstm_mean["pinball"],
}).reset_index()

def verdict(r):
    gap = r["skill_lstm"] - r["skill_lgbm"]
    if abs(gap) <= (r["skill_lstm_std"] or 0.0):
        return "tie (within noise)"
    return "lstm" if gap > 0 else "lightgbm"

table["winner"] = table.apply(verdict, axis=1)
table = table.round(3)
table.to_csv(OUT / "dl_vs_gbm_comparison.csv", index=False)
table

In [ ]:
wins = table["winner"].value_counts()
print("Winner by (target, horizon):")
print(wins.to_string())
print()
print(f"Compute: LightGBM {lgbm_secs:.0f}s (CPU) vs LSTM {lstm_secs/N_SEEDS:.0f}s/seed on {DEVICE}.")
faster = lstm_secs / N_SEEDS / max(lgbm_secs, 1e-9)
print(f"The LSTM costs about {faster:.0f}x the LightGBM wall-clock per fit.")

### Skill-by-horizon, per target

In [ ]:
targets = table["target"].unique()
fig, axes = plt.subplots(1, len(targets), figsize=(5 * len(targets), 4), squeeze=False)
for ax, tgt in zip(axes[0], targets):
    sub = table[table["target"] == tgt]
    ax.plot(sub["horizon_day"], sub["skill_lgbm"], "o-", label="LightGBM")
    ax.errorbar(sub["horizon_day"], sub["skill_lstm"], yerr=sub["skill_lstm_std"],
                fmt="s-", capsize=3, label="LSTM (mean +/- std)")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(tgt.replace("feat_", "")); ax.set_xlabel("horizon (days)")
    ax.set_ylabel("skill vs persistence"); ax.legend()
plt.tight_layout(); plt.savefig(OUT / "dl_vs_gbm_skill.png", dpi=120); plt.show()

## 5. Forecast fan, both models (headline target, one asset)

A visual sanity check: the p10/p50/p90 fan forecast forward from the most recent origin, for both models, against what actually happened.

In [ ]:
TGT = TARGETS[0]
sup = build_supervised_frame(panel, TGT, config)
cut = sup["date"].quantile(0.9)
train = sup[sup["date"] <= cut]
gbm_m = LightGBMForecaster(config).fit(train, TGT)
lstm_m = LSTMForecaster(config).fit(train, TGT)

asset = sup["asset_id"].iloc[0]
origin = train[train["asset_id"] == asset].sort_values("date").tail(1)
o_date = origin["date"].iloc[0]
gp = gbm_m.predict(origin, TGT).sort_values("horizon_day")
lp = lstm_m.predict(origin, TGT).sort_values("horizon_day")

actual = sup[(sup["asset_id"] == asset) & (sup["date"] > o_date)].sort_values("date").head(HORIZON)

fig, ax = plt.subplots(figsize=(8, 4.5))
for p, name, c in [(gp, "LightGBM", "tab:blue"), (lp, "LSTM", "tab:orange")]:
    ax.fill_between(p["horizon_day"], p["p10"], p["p90"], alpha=0.18, color=c)
    ax.plot(p["horizon_day"], p["p50"], "o-", color=c, label=f"{name} p50 (p10-p90 band)")
if not actual.empty:
    ax.plot(range(1, len(actual) + 1), actual[TGT].to_numpy(), "k--x", label="actual")
ax.set_title(f"{TGT.replace('feat_', '')} - asset {asset}, origin {pd.Timestamp(o_date).date()}")
ax.set_xlabel("horizon (days)"); ax.set_ylabel(TGT); ax.legend()
plt.tight_layout(); plt.savefig(OUT / "dl_vs_gbm_fan.png", dpi=120); plt.show()

## Takeaways (fill in from the run)

- **Who won, and where.** Read the `winner` column and the skill plot. Expect LightGBM to be hard to beat at short horizons; look for any LSTM edge at longer horizons or on `skill_vs_persistence_events` (the active periods that matter operationally).
- **Is the gap real?** A win inside the LSTM's seed-to-seed std band is noise, not a result. The table marks those `tie (within noise)`.
- **Cost.** The LSTM's wall-clock multiple over LightGBM is printed above. A tie at many times the cost argues for shipping LightGBM.
- **Decision.** Ship the single better model. If it is a tie, ship LightGBM (simpler, faster, no GPU, already integrated into `climagrid.forecast`). If the LSTM wins clearly and beyond noise, wire it into the load-and-serve path and ship it instead.

This is an honest bake-off: the harness, splits, quantiles and conformal calibration are identical, and a negative or tied result is reported as plainly as a win.